# Estimate planned-shift throughput
Use information assumed available before a shift. Do not include actual_tonnes or energy_kwh as features. We hold out the final 10 days rather than randomly mixing future and past rows. This remains synthetic practice, not a validated operational forecast.

In [ ]:
import re
# EDIT CATALOG if the detected default is not a writable Unity Catalog catalog.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "iman_mining_lab"
VOLUME = "project_files"
for identifier in [CATALOG, SCHEMA, VOLUME]:
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier), "Use simple catalog/schema names"
assert CATALOG not in ["hive_metastore", "system"], "Choose a writable Unity Catalog catalog in CATALOG"
PREFIX = f"`{CATALOG}`.`{SCHEMA}`"
FILES = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
def table(name):
    assert re.fullmatch(r"[a-z_]+", name)
    return f"{PREFIX}.`{name}`"
print("Catalog:", CATALOG, "Schema:", SCHEMA, "Volume path:", FILES)


In [ ]:
import pandas as pd
source=spark.table(table("silver_shifts"))
assert source.count() <= 10000, "Do not collect large production tables to the driver"
pdf=source.orderBy("shift_date","shift_id").toPandas()
pdf["shift_date"]=pd.to_datetime(pdf["shift_date"]).dt.strftime("%Y-%m-%d")
assert len(pdf)<=10000, "Do not collect large production tables to the driver"


In [ ]:
from mining_core import FEATURES, fit_model
model,metrics,train,test,pred=fit_model(pdf)
y_test=test.actual_tonnes
assert len(train)==400 and len(test)==200
print(metrics)
print("MAE is error in tonnes; compare the model with the baseline.")


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(5,5));ax.scatter(y_test,pred,alpha=.6)
lo=min(y_test.min(),pred.min());hi=max(y_test.max(),pred.max())
ax.plot([lo,hi],[lo,hi],"r--");ax.set(xlabel="Actual tonnes",ylabel="Predicted tonnes",title="Held-out final 10 days");plt.show()
importance=pd.DataFrame({"feature":FEATURES,"importance":model.feature_importances_}).sort_values("importance",ascending=False)
display(importance)


In [ ]:
future=pd.read_csv(f"{FILES}/next_shifts.csv")
assert set(FEATURES).issubset(future.columns)
assert future[FEATURES].notna().all().all()
scored=future[["shift_id","shift_date","machine_id"]].copy()
scored["predicted_tonnes"]=model.predict(future[FEATURES]).round(2)
schema="shift_id STRING, shift_date STRING, machine_id STRING, predicted_tonnes DOUBLE"
spark.createDataFrame(list(scored.itertuples(index=False,name=None)),schema).write.format("delta").mode("overwrite").option("overwriteSchema",True).saveAsTable(table("predicted_shifts"))
display(scored)
assert len(scored)==10


## Optional experiment tracking
This final cell logs parameters and metrics only; it does not register or deploy the model. Set RUN_MLFLOW=True only when you want to try it. The code uses your personal workspace experiment. If experiment permission fails, keep the displayed metrics and continue; core results do not depend on MLflow.

In [ ]:
RUN_MLFLOW = False
if RUN_MLFLOW:
    import mlflow, importlib.metadata as md
    user=spark.sql("SELECT current_user()").first()[0]
    mlflow.set_experiment(f"/Users/{user}/mining_throughput_practice")
    with mlflow.start_run(run_name="temporal_holdout_rf"):
        mlflow.log_params({"train_rows":len(train),"test_rows":len(test),"cutoff":"2026-09-21","n_estimators":100,"max_depth":8,"random_state":42,"sklearn_version":md.version("scikit-learn")})
        mlflow.log_metrics(metrics)
        mlflow.set_tag("data_origin","synthetic_learning_only")
        print("Logged run",mlflow.active_run().info.run_id)


## Finish
Your schema now contains predicted_shifts with 10 rows. Explain the baseline, date split and MAE. Feature importance is model association, not causal evidence. Do not claim this model predicts failures. Stop compute after exporting useful work. See START_HERE.html for the shutdown procedure.